# Experiment 4 :: Term Frequency, NER and TF-IDF

- 4.1 Term frequency and named entity recognition using toolkits (NLTK, spaCy)
- 4.2 Term frequency without any toolkit
- 4.3 TF, DF, IDF and TF-IDF computed from scratch across three documents

In [1]:
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

zsh:1: command not found: pip


zsh:1: command not found: python


In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/apple/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/apple/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Input data

In [3]:
text = open('input_tf_ner_data.txt', encoding='utf-8').read()
flat = ' '.join(text.split())

documents = [
    'Natural language processing is a field of artificial intelligence.',
    'Natural language processing helps computers understand human language.',
    'Machine learning is an important part of artificial intelligence.',
]

print('Characters:', len(text))
print('Words:', len(flat.split()))
print()
print(flat[:300] + '...')

Characters: 2004
Words: 301

Natural language processing has a long history that begins well before modern computers. In 1950, Alan Turing published a paper in the journal Mind that proposed what is now called the Turing test, a way to judge whether a machine can hold a conversation that is indistinguishable from a human one. A...


## 4.1 Term frequency and NER using toolkits

### 4.1 a. Named entity recognition with spaCy

In [4]:
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(flat)

print(f"{'ENTITY':<42}{'LABEL':<14}{'MEANING'}")
for ent in doc.ents:
    print(f'{ent.text:<42}{ent.label_:<14}{spacy.explain(ent.label_)}')

print()
print('Total entities:', len(doc.ents))

/private/tmp/claude-501/-Users-apple/40dd8ba3-bf33-410e-9beb-76b2b19fde63/scratchpad/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ENTITY                                    LABEL         MEANING
1950                                      DATE          Absolute or relative dates or periods
Alan Turing                               PERSON        People, including fictional
Mind                                      PRODUCT       Objects, vehicles, foods, etc. (not services)
A few years later                         DATE          Absolute or relative dates or periods
1957                                      DATE          Absolute or relative dates or periods
Noam Chomsky                              PERSON        People, including fictional
the Massachusetts Institute of Technology ORG           Companies, agencies, institutions, etc.
Syntactic Structures                      ORG           Companies, agencies, institutions, etc.
the United States                         GPE           Countries, cities, states
1966                                      DATE          Absolute or relative dates or periods
ALPAC           

### 4.1 b. Term frequency with NLTK

In [5]:
from collections import Counter
from nltk.tokenize import word_tokenize

tokens = [token.lower() for token in word_tokenize(flat) if token.isalnum()]
tf = Counter(tokens)

with open('4_1_tf.csv', 'w', encoding='utf-8') as f:
    f.write('term,frequency\n')
    for term, freq in tf.most_common():
        f.write(f'{term},{freq}\n')

print('Total tokens:', len(tokens))
print('Unique terms:', len(tf))
print()
print(f"{'TERM':<16}{'FREQUENCY'}")
for term, freq in tf.most_common(10):
    print(f'{term:<16}{freq}')

Total tokens: 299
Unique terms: 196

TERM            FREQUENCY
the             17
in              14
a               10
of              9
and             9
language        6
that            6
one             4
natural         3
processing      3


## 4.2 Term frequency without a toolkit

In [6]:
PUNCTUATION = r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""

clean = ''.join(c for c in flat.lower() if c not in PUNCTUATION)
manual_tokens = clean.split()

manual_tf = {}
for term in manual_tokens:
    manual_tf[term] = manual_tf.get(term, 0) + 1

ranked = sorted(manual_tf.items(), key=lambda item: (-item[1], item[0]))

with open('4_2_tf.csv', 'w', encoding='utf-8') as f:
    f.write('term,frequency\n')
    for term, freq in ranked:
        f.write(f'{term},{freq}\n')

print('Total tokens:', len(manual_tokens))
print('Unique terms:', len(manual_tf))
print()
print(f"{'TERM':<16}{'FREQUENCY'}")
for term, freq in ranked[:10]:
    print(f'{term:<16}{freq}')

Total tokens: 301
Unique terms: 198

TERM            FREQUENCY
the             17
in              14
a               10
and             9
of              9
language        6
that            6
one             4
field           3
from            3


In [7]:
disagreements = {term for term in set(tf) | set(manual_tf)
                 if tf.get(term, 0) != manual_tf.get(term, 0)}

print('Terms where the two methods disagree:', len(disagreements))
for term in sorted(disagreements):
    print(f'  {term}: nltk={tf.get(term, 0)} manual={manual_tf.get(term, 0)}')

Terms where the two methods disagree: 2
  gpt2: nltk=0 manual=1
  gpt3: nltk=0 manual=1


## 4.3 TF-IDF without a toolkit

TF is the raw count of a term in one document. DF is the number of documents the term
appears in. IDF is log10(N / DF) with N = 3 documents, so a term found in every document
gets IDF = 0 and its TF-IDF vanishes no matter how often it occurs.

In [8]:
import math

processed = []
for document in documents:
    doc_clean = ''.join(c for c in document.lower() if c not in PUNCTUATION)
    processed.append(doc_clean.split())

tfs = []
for doc_tokens in processed:
    counts = {}
    for term in doc_tokens:
        counts[term] = counts.get(term, 0) + 1
    tfs.append(counts)

df = {}
for counts in tfs:
    for term in counts:
        df[term] = df.get(term, 0) + 1

N = len(documents)
idf = {term: math.log10(N / count) for term, count in df.items()}

print('Vocabulary size:', len(df))
print()
print(f"{'TERM':<16}{'DF':<5}{'IDF'}")
for term in sorted(df, key=lambda t: (df[t], t)):
    print(f'{term:<16}{df[term]:<5}{idf[term]:.4f}')

Vocabulary size: 18

TERM            DF   IDF
a               1    0.4771
an              1    0.4771
computers       1    0.4771
field           1    0.4771
helps           1    0.4771
human           1    0.4771
important       1    0.4771
learning        1    0.4771
machine         1    0.4771
part            1    0.4771
understand      1    0.4771
artificial      2    0.1761
intelligence    2    0.1761
is              2    0.1761
language        2    0.1761
natural         2    0.1761
of              2    0.1761
processing      2    0.1761


In [9]:
for i, counts in enumerate(tfs):
    print(f'Document {i + 1} :: {documents[i]}')
    print(f"{'TERM':<16}{'TF':<5}{'DF':<5}{'IDF':<10}{'TF-IDF'}")
    scored = sorted(counts.items(), key=lambda item: -item[1] * idf[item[0]])
    for term, count in scored:
        print(f'{term:<16}{count:<5}{df[term]:<5}{idf[term]:<10.4f}{count * idf[term]:.4f}')
    print()

Document 1 :: Natural language processing is a field of artificial intelligence.
TERM            TF   DF   IDF       TF-IDF
a               1    1    0.4771    0.4771
field           1    1    0.4771    0.4771
natural         1    2    0.1761    0.1761
language        1    2    0.1761    0.1761
processing      1    2    0.1761    0.1761
is              1    2    0.1761    0.1761
of              1    2    0.1761    0.1761
artificial      1    2    0.1761    0.1761
intelligence    1    2    0.1761    0.1761

Document 2 :: Natural language processing helps computers understand human language.
TERM            TF   DF   IDF       TF-IDF
helps           1    1    0.4771    0.4771
computers       1    1    0.4771    0.4771
understand      1    1    0.4771    0.4771
human           1    1    0.4771    0.4771
language        2    2    0.1761    0.3522
natural         1    2    0.1761    0.1761
processing      1    2    0.1761    0.1761

Document 3 :: Machine learning is an important part of art

In [10]:
for i, counts in enumerate(tfs):
    scored = sorted(counts.items(), key=lambda item: -item[1] * idf[item[0]])
    top = [f'{term} ({count * idf[term]:.4f})' for term, count in scored[:3]]
    print(f'Document {i + 1}:', ', '.join(top))

Document 1: a (0.4771), field (0.4771), natural (0.1761)
Document 2: helps (0.4771), computers (0.4771), understand (0.4771)
Document 3: machine (0.4771), learning (0.4771), an (0.4771)


The top TF-IDF terms are the distinguishing words of each document: `field` for the first,
`helps`, `computers`, `understand` and `human` for the second, `machine` and `learning` for
the third. No term appears in all three documents, so nothing scores exactly zero, but the
shared vocabulary (`natural`, `language`, `processing`, `is`, `of`, `artificial`,
`intelligence`) drops to the 0.1761 band while document-specific words score 0.4771, almost
three times higher. The ranking also puts `a` and `an` at the top simply because each is
unique to one document, a reminder that TF-IDF measures distinctiveness across the
collection, not importance within the language.